Подготовка (загрузка данных)

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
import kagglehub
import os

spark = SparkSession.builder \
    .appName("BikeShareAnalysis") \
    .getOrCreate()

path = kagglehub.dataset_download("benhamner/sf-bay-area-bike-share")

print("Path to dataset files:", path)

for root, dirs, files in os.walk(path):
    for file in files:
        print(os.path.join(root, file))

# Загрузка данных
trips_path = os.path.join(path, "trip.csv")
stations_path = os.path.join(path, "station.csv")

trips = spark.read.csv(trips_path, header=True, inferSchema=True)
stations = spark.read.csv(stations_path, header=True, inferSchema=True)

trips.printSchema()
stations.printSchema()

Using Colab cache for faster access to the 'sf-bay-area-bike-share' dataset.
Path to dataset files: /kaggle/input/sf-bay-area-bike-share
/kaggle/input/sf-bay-area-bike-share/status.csv
/kaggle/input/sf-bay-area-bike-share/weather.csv
/kaggle/input/sf-bay-area-bike-share/station.csv
/kaggle/input/sf-bay-area-bike-share/trip.csv
/kaggle/input/sf-bay-area-bike-share/database.sqlite
root
 |-- id: integer (nullable = true)
 |-- duration: integer (nullable = true)
 |-- start_date: string (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: integer (nullable = true)
 |-- end_date: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: integer (nullable = true)
 |-- bike_id: integer (nullable = true)
 |-- subscription_type: string (nullable = true)
 |-- zip_code: string (nullable = true)

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (n

1. Велосипед с максимальным временем пробега

Суммируем duration по каждому bike_id

In [3]:
bike_total_time = trips.groupBy("bike_id") \
    .agg(sum("duration").alias("total_duration")) \
    .orderBy(desc("total_duration"))

bike_total_time.show(1)

+-------+--------------+
|bike_id|total_duration|
+-------+--------------+
|    535|      18611693|
+-------+--------------+
only showing top 1 row


2) Наибольшее геодезическое расстояние между станциями.
Используем формулу гаверсинуса (расстояние по сфере)

In [4]:
from pyspark.sql.functions import radians, sin, cos, sqrt, atan2

stations_a = stations.alias("a")
stations_b = stations.alias("b")

R = 6371  # радиус Земли в км

distances = stations_a.crossJoin(stations_b) \
    .filter(col("a.id") != col("b.id")) \
    .withColumn("lat1", radians(col("a.lat"))) \
    .withColumn("lon1", radians(col("a.long"))) \
    .withColumn("lat2", radians(col("b.lat"))) \
    .withColumn("lon2", radians(col("b.long"))) \
    .withColumn("dlat", col("lat2") - col("lat1")) \
    .withColumn("dlon", col("lon2") - col("lon1")) \
    .withColumn("a_val", sin(col("dlat")/2)**2 +
                cos(col("lat1")) * cos(col("lat2")) * sin(col("dlon")/2)**2) \
    .withColumn("c", 2 * atan2(sqrt(col("a_val")), sqrt(1 - col("a_val")))) \
    .withColumn("distance", col("c") * R)

distances.select("a.name", "b.name", "distance") \
    .orderBy(desc("distance")) \
    .show(1, truncate=False)

+--------------------------+----------------------+-----------------+
|name                      |name                  |distance         |
+--------------------------+----------------------+-----------------+
|SJSU - San Salvador at 9th|Embarcadero at Sansome|69.92087595428183|
+--------------------------+----------------------+-----------------+
only showing top 1 row


3) Путь велосипеда с максимальным временем пробега

Шаги:
 1. Найти bike_id с максимальным временем
 2. Отсортировать его поездки по времени
 3. Вывести маршрут

In [5]:
# 1. Найти велосипед
top_bike = bike_total_time.first()["bike_id"]

# 2. Путь велосипеда
bike_path = trips.filter(col("bike_id") == top_bike) \
    .orderBy("start_date") \
    .select("start_date", "start_station_name", "end_station_name", "duration")

bike_path.show(truncate=False)

+---------------+---------------------------------------------+---------------------------------------------+--------+
|start_date     |start_station_name                           |end_station_name                             |duration|
+---------------+---------------------------------------------+---------------------------------------------+--------+
|1/1/2014 13:42 |Mechanics Plaza (Market at Battery)          |Embarcadero at Sansome                       |3289    |
|1/1/2014 18:51 |Embarcadero at Sansome                       |Market at 4th                                |1286    |
|1/1/2014 19:48 |Market at 4th                                |South Van Ness at Market                     |795     |
|1/10/2014 20:13|Market at 10th                               |Powell Street BART                           |235     |
|1/10/2014 8:09 |Embarcadero at Folsom                        |San Francisco Caltrain (Townsend at 4th)     |596     |
|1/10/2014 8:21 |San Francisco Caltrain (Townsen

4) Количество велосипедов в системе

In [6]:
bike_count = trips.select("bike_id").distinct().count()
print("Количество велосипедов:", bike_count)

Количество велосипедов: 700


5) Пользователи, потратившие более 3 часов

In [7]:
users_over_3h = trips.groupBy("zip_code") \
    .agg(sum("duration").alias("total_duration")) \
    .filter(col("total_duration") > 10800) \
    .orderBy(desc("total_duration"))

users_over_3h.show()

+--------+--------------+
|zip_code|total_duration|
+--------+--------------+
|   94107|      49757162|
|     nil|      45725550|
|    NULL|      27723273|
|   94105|      25596128|
|   94133|      21637675|
|   94102|      19128021|
|   94103|      19127388|
|   95531|      17270400|
|   94111|      14244997|
|   95112|      12742370|
|   94109|      12057128|
|   94040|       7807926|
|   94110|       7421936|
|   94117|       6901313|
|   94301|       6590378|
|   94041|       6276284|
|   94158|       6248167|
|   94306|       5550643|
|   94025|       5178237|
|   94108|       5127562|
+--------+--------------+
only showing top 20 rows


In [8]:
trips = trips.withColumn("start_date", to_timestamp("start_date")) \
             .withColumn("end_date", to_timestamp("end_date"))

In [9]:
trips.cache()
stations.cache()

DataFrame[id: int, name: string, lat: double, long: double, dock_count: int, city: string, installation_date: string]